# SurfaceTarget-CRC — phase 01 — patch 1 22

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.22
#
# PATIENT-MATCHED COAD TUMOR–NORMAL ANALYSIS
#
# Objectives:
# 1. Identify Tier B patients with paired normal tissue
# 2. Validate patient and workflow correspondence
# 3. Fit a patient-adjusted differential expression model
# 4. Calculate descriptive within-patient expression changes
# 5. Evaluate robustness of existing screening candidates
# 6. Save reproducible results and provenance
#
# IMPORTANT:
# - This is a complementary analysis, not independent validation.
# - The paired cohort is small.
# - No samples or raw counts are permanently modified.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

GENES_PATH = (
    META / "tcga_gene_annotations.csv"
)

BASELINE_PATH = (
    RESULTS / "tcga_coad_qc_sensitivity_comparison.csv"
)

QC_PATH = (
    RESULTS / "tcga_full_cohort_qc.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.22 — PATIENT-MATCHED COAD ANALYSIS")
print("=" * 65)


# ------------------------------------------------------------
# 2. Validate dependencies and load data
# ------------------------------------------------------------

print("\nSoftware versions:")
print("PyDESeq2:", version("pydeseq2"))
print("Pandas:", pd.__version__)

if version("pydeseq2") != "0.5.4":
    raise RuntimeError("Unexpected PyDESeq2 version.")

for path in [
    MATRIX_PATH,
    SAMPLES_PATH,
    GENES_PATH,
    BASELINE_PATH,
    QC_PATH
]:
    if not path.exists():
        raise FileNotFoundError(path)

with np.load(MATRIX_PATH, allow_pickle=False) as archive:
    counts = archive["counts"]
    gene_ids = archive["gene_ids"]
    file_ids = archive["file_ids"]

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

baseline = pd.read_csv(BASELINE_PATH)

qc = pd.read_csv(
    QC_PATH,
    dtype=str,
    keep_default_na=False
)

if counts.shape != (60660, 472):
    raise RuntimeError("Unexpected matrix shape.")

if not np.array_equal(file_ids, samples["file_id"].to_numpy()):
    raise RuntimeError("Sample identifiers are misaligned.")

if not np.array_equal(gene_ids, genes["gene_id"].to_numpy()):
    raise RuntimeError("Gene identifiers are misaligned.")

if baseline["gene_id"].duplicated().any():
    raise RuntimeError("Duplicate baseline genes.")

print("\nData loaded and validated.")


# ------------------------------------------------------------
# 3. Identify the shared workflow
# ------------------------------------------------------------

normal_workflows = samples.loc[
    samples["dataset_group"].eq("Normal"),
    "workflow_version"
].unique()

if len(normal_workflows) != 1:
    raise RuntimeError("Unexpected normal workflow structure.")

SHARED_WORKFLOW = normal_workflows[0]

coad = samples[
    samples["project"].eq("TCGA-COAD")
    & samples["workflow_version"].eq(SHARED_WORKFLOW)
].copy()

tumors = coad[
    coad["dataset_group"].eq("Tumor")
    & coad["cohort_tier"].eq("Tier B")
].copy()

normals = coad[
    coad["dataset_group"].eq("Normal")
].copy()

paired_patients = sorted(
    set(tumors["patient_id"])
    & set(normals["patient_id"])
)

print("\n" + "-" * 65)
print("MATCHED PATIENT IDENTIFICATION")
print("-" * 65)

print("\nTier B tumors:", len(tumors))
print("Available normals:", len(normals))
print("Patients with both:", len(paired_patients))

if len(paired_patients) != 12:
    raise RuntimeError(
        "Expected 12 matched Tier B COAD patients. "
        "Review the cohort before proceeding."
    )


# ------------------------------------------------------------
# 4. Construct paired sample manifest
# ------------------------------------------------------------

pair_records = []

for patient_id in paired_patients:

    tumor_rows = tumors[
        tumors["patient_id"].eq(patient_id)
    ]

    normal_rows = normals[
        normals["patient_id"].eq(patient_id)
    ]

    if len(tumor_rows) != 1 or len(normal_rows) != 1:
        raise RuntimeError(
            f"Ambiguous paired samples: {patient_id}"
        )

    tumor = tumor_rows.iloc[0]
    normal = normal_rows.iloc[0]

    pair_records.append({
        "patient_id": patient_id,
        "tumor_file_id": tumor["file_id"],
        "normal_file_id": normal["file_id"],
        "tumor_sample_id": tumor["sample_id"],
        "normal_sample_id": normal["sample_id"],
        "workflow_version": SHARED_WORKFLOW
    })

pairs = pd.DataFrame(pair_records)

if pairs["patient_id"].duplicated().any():
    raise RuntimeError("Duplicate matched patients.")

pairs_path = (
    META / "tcga_coad_tierb_matched_pairs.csv"
)

pairs.to_csv(pairs_path, index=False)

print("\nMatched pairs:")
print(
    pairs[
        ["patient_id", "tumor_sample_id", "normal_sample_id"]
    ].to_string(index=False)
)

print("\nMatched-patient validation: PASSED")


# ------------------------------------------------------------
# 5. Prepare sample ordering
# ------------------------------------------------------------

# Each patient contributes:
# Normal sample followed by tumor sample.

ordered_file_ids = []

conditions = []

patient_labels = []

for _, row in pairs.iterrows():

    ordered_file_ids.extend([
        row["normal_file_id"],
        row["tumor_file_id"]
    ])

    conditions.extend(["Normal", "Tumor"])

    patient_labels.extend([
        row["patient_id"],
        row["patient_id"]
    ])

file_lookup = {
    file_id: index
    for index, file_id in enumerate(file_ids)
}

sample_positions = np.array([
    file_lookup[fid]
    for fid in ordered_file_ids
])

if len(sample_positions) != 24:
    raise RuntimeError("Unexpected paired sample count.")

if len(set(ordered_file_ids)) != 24:
    raise RuntimeError("Repeated RNA-seq file in paired analysis.")

print("\nPaired samples prepared:", len(sample_positions))


# ------------------------------------------------------------
# 6. Check existing QC flags
# ------------------------------------------------------------

qc_lookup = qc.set_index("file_id")

paired_qc = qc_lookup.loc[
    ordered_file_ids
].copy()

paired_qc["n_qc_review_flags"] = pd.to_numeric(
    paired_qc["n_qc_review_flags"],
    errors="raise"
)

n_flagged = int(
    (paired_qc["n_qc_review_flags"] > 0).sum()
)

print("\nPreviously QC-flagged paired samples:", n_flagged)

print(
    "No samples will be excluded from this paired analysis."
)


# ------------------------------------------------------------
# 7. Prepare protein-coding gene counts
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING PAIRED EXPRESSION MATRIX")
print("-" * 65)

coding_mask = (
    genes["gene_type"]
    .eq("protein_coding")
    .to_numpy(dtype=bool)
)

coding_gene_ids = gene_ids[coding_mask]

paired_counts = counts[
    coding_mask, :
][:, sample_positions]

# Library sizes from all annotated genes.

library_sizes = counts[
    :, sample_positions
].sum(axis=0)

if np.any(library_sizes <= 0):
    raise RuntimeError("Zero library size detected.")

paired_cpm = (
    paired_counts.astype(np.float64)
    / library_sizes[np.newaxis, :]
    * 1_000_000
)

# Exploratory expression filter:
# CPM >= 1 in at least 4 of 24 samples.

expressed_mask = (
    np.count_nonzero(
        paired_cpm >= 1,
        axis=1
    ) >= 4
)

selected_gene_ids = coding_gene_ids[
    expressed_mask
]

selected_counts = paired_counts[
    expressed_mask, :
]

selected_cpm = paired_cpm[
    expressed_mask, :
]

if len(selected_gene_ids) < 5000:
    raise RuntimeError(
        "Unexpectedly few expressed genes."
    )

print("\nProtein-coding genes:", len(coding_gene_ids))
print("Genes passing paired expression filter:", len(selected_gene_ids))
print("Paired samples:", selected_counts.shape[1])


# ------------------------------------------------------------
# 8. Descriptive within-patient expression changes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("WITHIN-PATIENT EXPRESSION ASSESSMENT")
print("-" * 65)

# Sample order:
# Normal, Tumor, Normal, Tumor, ...
#
# Calculate log2((tumor CPM + 1) / (normal CPM + 1)).
#
# This is descriptive and not the DESeq2 model estimate.

normal_cpm = selected_cpm[:, 0::2]
tumor_cpm = selected_cpm[:, 1::2]

paired_log2_ratio = np.log2(
    (tumor_cpm + 1.0)
    / (normal_cpm + 1.0)
)

descriptive = pd.DataFrame({

    "gene_id": selected_gene_ids,

    "paired_median_log2_cpm_ratio":
        np.median(paired_log2_ratio, axis=1),

    "paired_fraction_tumor_higher":
        np.mean(paired_log2_ratio > 0, axis=1),

    "paired_fraction_log2ratio_gt1":
        np.mean(paired_log2_ratio > 1, axis=1),

    "paired_median_tumor_cpm":
        np.median(tumor_cpm, axis=1),

    "paired_median_normal_cpm":
        np.median(normal_cpm, axis=1)
})

print("\nDescriptive paired expression metrics calculated.")


# ------------------------------------------------------------
# 9. Prepare PyDESeq2 inputs
# ------------------------------------------------------------

counts_df = pd.DataFrame(
    selected_counts.T,
    index=ordered_file_ids,
    columns=selected_gene_ids
)

metadata = pd.DataFrame({

    "patient_id": patient_labels,

    "condition": conditions

}, index=ordered_file_ids)

metadata["patient_id"] = pd.Categorical(
    metadata["patient_id"],
    categories=paired_patients
)

metadata["condition"] = pd.Categorical(
    metadata["condition"],
    categories=["Normal", "Tumor"],
    ordered=True
)

# Preserve AnnData compatibility.

counts_df.index = pd.Index(
    counts_df.index,
    dtype="string"
)

counts_df.columns = pd.Index(
    counts_df.columns,
    dtype="string"
)

metadata.index = pd.Index(
    metadata.index,
    dtype="string"
)

if not counts_df.index.equals(metadata.index):
    raise RuntimeError("PyDESeq2 input alignment failed.")

# Confirm every patient contributes exactly
# one normal and one tumor sample.

pair_check = pd.crosstab(
    metadata["patient_id"],
    metadata["condition"]
)

if not (
    pair_check["Normal"].eq(1).all()
    and pair_check["Tumor"].eq(1).all()
):
    raise RuntimeError("Invalid paired design.")

print("\nPaired model input validation: PASSED")
print("Model input dimensions:", counts_df.shape)


# ------------------------------------------------------------
# 10. Fit patient-adjusted differential expression
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FITTING PATIENT-ADJUSTED DESEQ2 MODEL")
print("=" * 65)

print("\nDesign: ~ patient_id + condition")
print("Contrast: Tumor versus Normal")

# Disable Cook's outlier replacement because
# each patient-condition combination has only
# one sample in this small paired dataset.

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="~ patient_id + condition",
    refit_cooks=False,
    n_cpus=2
)

dds.deseq2()

print("\nPaired model fitting: COMPLETE")


# ------------------------------------------------------------
# 11. Calculate paired DE statistics
# ------------------------------------------------------------

stats = DeseqStats(
    dds,
    contrast=["condition", "Tumor", "Normal"],
    alpha=0.05,
    n_cpus=2
)

stats.summary()

paired_results = (
    stats.results_df
    .copy()
    .rename_axis("gene_id")
    .reset_index()
)

if paired_results["gene_id"].duplicated().any():
    raise RuntimeError("Duplicate paired-result genes.")

paired_results = paired_results.merge(
    descriptive,
    on="gene_id",
    how="left",
    validate="one_to_one"
)

paired_results = paired_results.merge(
    genes[["gene_id", "gene_name"]],
    on="gene_id",
    how="left",
    validate="one_to_one"
)

paired_results["paired_upregulated"] = (
    paired_results["padj"].lt(0.05)
    & paired_results["log2FoldChange"].gt(1)
)

print("\nPaired DE statistics: COMPLETE")


# ------------------------------------------------------------
# 12. Compare with existing screening candidates
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CROSS-ANALYSIS GENE SUPPORT")
print("=" * 65)

required_baseline_columns = {
    "gene_id",
    "gene_name",
    "baseline_log2fc",
    "robust_screening_candidate"
}

if not required_baseline_columns.issubset(
    baseline.columns
):
    raise RuntimeError(
        "Unexpected baseline sensitivity table format."
    )

baseline["robust_screening_candidate"] = (
    baseline["robust_screening_candidate"]
    .astype(str)
    .str.lower()
    .eq("true")
)

comparison = baseline.merge(
    paired_results[
        [
            "gene_id",
            "log2FoldChange",
            "padj",
            "paired_median_log2_cpm_ratio",
            "paired_fraction_tumor_higher",
            "paired_fraction_log2ratio_gt1",
            "paired_upregulated"
        ]
    ],
    on="gene_id",
    how="left",
    validate="one_to_one"
)

comparison = comparison.rename(
    columns={
        "log2FoldChange": "paired_model_log2fc",
        "padj": "paired_model_padj"
    }
)

# Descriptive support requires:
# - Tumor CPM greater than normal in >= 75% of pairs
# - Median paired log2 CPM ratio > 1

comparison["paired_direction_supported"] = (
    comparison["paired_fraction_tumor_higher"].ge(0.75)
    & comparison["paired_median_log2_cpm_ratio"].gt(1)
)

comparison["combined_transcriptomic_support"] = (
    comparison["robust_screening_candidate"]
    & comparison["paired_direction_supported"]
)

robust_candidates = comparison[
    comparison["robust_screening_candidate"]
].copy()

supported_candidates = robust_candidates[
    robust_candidates["paired_direction_supported"]
].copy()

print("\nPreviously robust screening genes:", len(robust_candidates))

print(
    "Genes with additional paired directional support:",
    len(supported_candidates)
)

print(
    "Genes significant in the paired model:",
    int(paired_results["paired_upregulated"].sum())
)

print("\nPaired directional support is descriptive,")
print("not an independent statistical validation.")


# ------------------------------------------------------------
# 13. Examine original leading genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TOP 20 GENE SUPPORT")
print("-" * 65)

top20 = robust_candidates.sort_values(
    "baseline_padj"
).head(20).copy()

display_columns = [
    "gene_name",
    "baseline_log2fc",
    "paired_model_log2fc",
    "paired_model_padj",
    "paired_median_log2_cpm_ratio",
    "paired_fraction_tumor_higher",
    "paired_direction_supported"
]

print(
    top20[display_columns]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)


# ------------------------------------------------------------
# 14. Generate comparison figure
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING MATCHED-PAIR FIGURE")
print("-" * 65)

plot_data = robust_candidates.dropna(
    subset=[
        "baseline_log2fc",
        "paired_model_log2fc"
    ]
)

fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(
    plot_data["baseline_log2fc"],
    plot_data["paired_model_log2fc"],
    s=12,
    alpha=0.4
)

ax.axhline(0, linestyle="--", linewidth=0.8)
ax.axvline(0, linestyle="--", linewidth=0.8)

ax.set_xlabel("Original discovery log2 fold change")
ax.set_ylabel("Patient-adjusted paired log2 fold change")

ax.set_title(
    "TCGA-COAD — Matched-Pair Expression Support"
)

ax.grid(alpha=0.2)

fig.tight_layout()

figure_path = (
    FIGURES / "tcga_coad_matched_pair_expression_support.png"
)

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nMatched-pair figure saved.")


# ------------------------------------------------------------
# 15. Save results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING MATCHED-PAIR RESULTS")
print("-" * 65)

paired_results.to_csv(
    RESULTS / "tcga_coad_paired_deseq2_results.csv",
    index=False
)

comparison.to_csv(
    RESULTS / "tcga_coad_paired_support_comparison.csv",
    index=False
)

supported_candidates.to_csv(
    RESULTS / "tcga_coad_paired_supported_candidates.csv",
    index=False
)

top20.to_csv(
    RESULTS / "tcga_coad_paired_top20_review.csv",
    index=False
)

metadata.reset_index().rename(
    columns={"index": "file_id"}
).to_csv(
    META / "tcga_coad_paired_model_metadata.csv",
    index=False
)


# ------------------------------------------------------------
# 16. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.22",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis": "Patient-matched Tier B COAD tumor-normal",

    "software": "PyDESeq2",

    "software_version": version("pydeseq2"),

    "pandas_version": pd.__version__,

    "patients": len(paired_patients),

    "tumor_samples": len(paired_patients),

    "normal_samples": len(paired_patients),

    "workflow_version": SHARED_WORKFLOW,

    "design": "~ patient_id + condition",

    "contrast": ["condition", "Tumor", "Normal"],

    "count_column": "unstranded",

    "expression_filter": (
        "CPM >= 1 in at least 4 of 24 samples"
    ),

    "genes_tested": len(paired_results),

    "refit_cooks": False,

    "qc_flagged_samples_included": n_flagged,

    "previously_robust_candidates": len(
        robust_candidates
    ),

    "paired_direction_supported_candidates": len(
        supported_candidates
    ),

    "limitations": [
        "Only 12 matched patients are available.",
        "This is not an independent validation cohort.",
        "Normal samples may be tumor-adjacent.",
        "Patient matching does not eliminate tissue-composition effects.",
        "Small-sample statistical estimates may be unstable.",
        "No clinical covariates beyond patient identity are included.",
        "CPM ratios are descriptive.",
        "No therapeutic conclusions are established."
    ]
}

with open(
    META / "tcga_coad_paired_analysis_provenance.json",
    "w"
) as f:
    json.dump(provenance, f, indent=4)


# ------------------------------------------------------------
# 17. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.22 SUMMARY")
print("=" * 65)

print("\nMatched patients:", len(paired_patients))

print("Paired RNA-seq samples:", len(sample_positions))

print("Genes tested:", len(paired_results))

print(
    "Previously robust screening candidates:",
    len(robust_candidates)
)

print(
    "Candidates with paired directional support:",
    len(supported_candidates)
)

print("\nFiles saved:")
print("1. tcga_coad_tierb_matched_pairs.csv")
print("2. tcga_coad_paired_deseq2_results.csv")
print("3. tcga_coad_paired_support_comparison.csv")
print("4. tcga_coad_paired_supported_candidates.csv")
print("5. tcga_coad_paired_top20_review.csv")
print("6. tcga_coad_paired_model_metadata.csv")
print("7. tcga_coad_paired_analysis_provenance.json")

print("\nFigure saved:")
print("tcga_coad_matched_pair_expression_support.png")

print("\nNo raw counts modified.")
print("Original analyses preserved.")
print("No permanent sample exclusions.")

print("\nPATCH 1.22 COMPLETE")
print("=" * 65)